# MACE NLU Behavior Shaping Trainer (Llama 3.2 1B)

Google Colab version of the behavior shaping trainer.
Uses Unsloth for ultra-fast fine-tuning on a T4 GPU.

**Instructions:**
1. Go to **Runtime -> Change runtime type** and select a **T4 GPU**.
2. Run the cells sequentially.
3. When prompted, upload your `generated_training_data.jsonl` file to the Colab environment.

In [ ]:
# Install Unsloth, TRL, and dependencies
%%capture
!pip install "unsloth[colab-new]" @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets

In [ ]:
import torch
import os
import json
import inspect
from datasets import Dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from google.colab import files

## Configuration
Set up your training parameters. Ensure your data file is uploaded to the root directory.

In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
DATA_FILE = "generated_training_data.jsonl"
OUTPUT_DIR = "mace_nlu_llama3.2_1b"
MAX_SEQ_LENGTH = 512
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

## Upload Dataset
Run this cell and use the upload button to upload `generated_training_data.jsonl` to the Colab runtime environment.

In [ ]:
if not os.path.exists(DATA_FILE):
    print(f"Upload {DATA_FILE}:")
    uploaded = files.upload()
else:
    print(f"{DATA_FILE} is already present.")

In [ ]:
def formatting_prompts_func(examples):
    inputs = examples["text"]
    outputs = []
    
    for i in range(len(inputs)):
        obj = {
            "text": examples["text"][i],
            "root_intent": examples["root_intent"][i],
            "memory_type": examples["memory_type"][i],
            "complexity": examples["complexity"][i],
            "entities": examples["entities"][i]
        }
        # Ensure formatting matches parsing grammar later
        json_str = json.dumps(obj, ensure_ascii=False)
        outputs.append(json_str)

    texts = []
    for input_text, output_json in zip(inputs, outputs):
        # Standard instruction/output formatting
        text = f"Input: {input_text}\nOutput: {output_json}" + "<|eot_id|>"
        texts.append(text)
    return {"text": texts}

## Initialize Model and Tokenizer
Loads the model in 4-bit precision with LoRA adapters for parameter-efficient fine-tuning.

In [ ]:
print(f"🚀 Loading model: {MODEL_NAME}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## Load and Prepare Dataset

In [ ]:
print(f"📂 Loading data from {DATA_FILE}...")
data = []
try:
    with open(DATA_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
except FileNotFoundError:
    print(f"❌ Error: {DATA_FILE} not found. Make sure you uploaded it correctly in the 'Upload Dataset' cell.")
    raise

dataset = Dataset.from_list(data)
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"📊 Training on {len(dataset)} examples")

## Fine-tune the Model
We use `DataCollatorForCompletionOnlyLM` to ensure the model only calculates loss on the generated JSON output, not on the input prompt.

In [ ]:
# Setup Collator to mask out the prompt
response_template = "Output:"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

# Handle TRL backwards compatibility for SFTTrainer
sft_args = {}
sig = inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters:
    sft_args["processing_class"] = tokenizer
else:
    sft_args["tokenizer"] = tokenizer

print(f"🔧 System args applied: {list(sft_args.keys())}")

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    data_collator = collator,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
    **sft_args
)

print("🔥 Starting training...")
trainer.train()

## Save, Convert to GGUF, and Export
This step saves the fine-tuned LoRA weights and attempts to compile a quantized GGUF file suitable for `ollama` or `llama.cpp`.

In [ ]:
print("💾 Saving model...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("📦 Converting to GGUF (q4_k_m)...")
try:
    model.save_pretrained_gguf(OUTPUT_DIR, tokenizer, quantization_method = "q4_k_m")
except Exception as e:
    print(f"⚠️ GGUF conversion failed: {e}")

print(f"✅ Done! Model saved to {OUTPUT_DIR}")

In [ ]:
import shutil

print(f"🗜️ Zipping {OUTPUT_DIR} for download...")
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)

print("⬇️ Prompting browser download (this may take a minute)...")
files.download(f"{OUTPUT_DIR}.zip")